To demonstrate the full machine learning pipeline—**EDA, Feature Engineering, Model Training, and Model Evaluation**—we'll predict house prices based on a dataset containing numerical, categorical, and missing values.

---

**Problem Statement**

Predict `Price` given four input features: `SqFt` (numeric), `Bedrooms` (numeric with missing values), `Location` (categorical), and `Age` (numeric).

---

**1. Exploratory Data Analysis (EDA)**

We load the dataset, check structural integrity, inspect missing values, and analyze summary statistics and feature distributions.

In [1]:
import numpy as np
import pandas as pd

# 1. Create a raw dataset with missing values and categorical data
raw_data = {
    'SqFt': [1500, 2100, 1200, 2800, 1800, 2500, np.nan, 3000, 1400, 2200],
    'Bedrooms': [3, 4, 2, np.nan, 3, 4, 2, 5, 2, 3],
    'Location': ['Suburbs', 'City', 'Suburbs', 'Downtown', 'Suburbs', 'Downtown', 'Suburbs', 'Downtown', 'Suburbs', 'City'],
    'Age': [10, 5, 20, 2, 15, 8, 12, 1, 25, 6],
    'Price': [250000, 380000, 180000, 520000, 290000, 460000, 210000, 580000, 190000, 400000]
}
df = pd.DataFrame(raw_data)

# 2. Inspect shape and data types
print("--- Data Structure ---")
print(df.info())

# 3. Check for missing values
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# 4. Summary statistics
print("\n--- Numerical Summary ---")
print(df.describe())

--- Data Structure ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SqFt      9 non-null      float64
 1   Bedrooms  9 non-null      float64
 2   Location  10 non-null     object 
 3   Age       10 non-null     int64  
 4   Price     10 non-null     int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 532.0+ bytes
None

--- Missing Values Count ---
SqFt        1
Bedrooms    1
Location    0
Age         0
Price       0
dtype: int64

--- Numerical Summary ---
              SqFt  Bedrooms        Age          Price
count     9.000000  9.000000  10.000000      10.000000
mean   2055.555556  3.111111  10.400000  346000.000000
std     632.675097  1.054093   7.763161  143310.075632
min    1200.000000  2.000000   1.000000  180000.000000
25%    1500.000000  2.000000   5.250000  220000.000000
50%    2100.000000  3.000000   9.000000  335000.000000
75%   

### Explanation of Step 1: Exploratory Data Analysis (EDA)
In this step, we perform initial exploration on our raw dataset:
* **Data Creation**: We construct a raw dataset of 10 houses using a pandas DataFrame, incorporating missing values (`np.nan`) in `SqFt` and `Bedrooms` to simulate real-world data.
* **`df.info()`**: Inspects structural details, verifying the shape of the DataFrame and checking data types (e.g., categorical objects and numerical values).
* **`df.isnull().sum()`**: Counts the exact number of missing values (`NaN` entries) in each column to pinpoint which features need imputation.
* **`df.describe()`**: Generates summary statistics (mean, standard deviation, min, max, and percentiles) for all numerical columns to understand the distribution and identify possible outliers.

---

**2. Feature Engineering**

We handle missing values (imputation), encode categorical strings into numbers (One-Hot Encoding), and scale numerical values so features share a common range.

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Separate features (X) and target variable (y)
X = df.drop(columns=['Price'])
y = df['Price']

# Define feature types
num_features = ['SqFt', 'Bedrooms', 'Age']
cat_features = ['Location']

# 1. Pipeline for numerical features: Impute missing values with median, then scale
# 2. Pipeline for categorical features: One-Hot Encode location values
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

# Fit and transform feature matrix
X_processed = preprocessor.fit_transform(X)

# Handle residual NaNs from imputation if using standalone transformers
imputer = SimpleImputer(strategy='median')
X_imputed_num = imputer.fit_transform(X[num_features])
scaler = StandardScaler()
X_scaled_num = scaler.fit_transform(X_imputed_num)

encoder = OneHotEncoder(drop='first', sparse_output=False)
X_encoded_cat = encoder.fit_transform(X[cat_features])

# Combine processed numerical and categorical arrays
X_final = np.hstack((X_scaled_num, X_encoded_cat))
print("Processed Feature Shape:", X_final.shape)

Processed Feature Shape: (10, 5)


### Explanation of Step 2: Feature Engineering
To make our raw features suitable for machine learning algorithms, we apply several preprocessing techniques:
* **Feature and Target Split**: We split the dataset into independent variables (features in `X`) and the dependent variable (target `Price` in `y`).
* **Numerical Imputation**: A `SimpleImputer` replaces missing values (`NaN`) in numerical columns with their calculated `median` values.
* **Numerical Scaling**: A `StandardScaler` standardizes the numerical features (`SqFt`, `Bedrooms`, and `Age`) to a mean of 0 and a standard deviation of 1. This prevents larger-scale variables from dominating the model.
* **Categorical Encoding**: An `OneHotEncoder` transforms the `Location` categories into numerical binary flags (dummy variables). Setting `drop='first'` helps avoid multicollinearity (the dummy variable trap).
* **`np.hstack`**: Finally, we concatenate the scaled numerical array and the encoded categorical array horizontally to form a unified, fully-processed feature matrix `X_final`.


---

**3. Model Training**

We split the processed dataset into training and test sets, select a Linear Regression algorithm, and train the model using `.fit()`.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# 1. Split into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

# 2. Instantiate the linear regression model
model = LinearRegression()

# 3. Train the model on training data
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


### Explanation of Step 3: Model Training
With our features clean and numeric, we proceed to train our predictor:
* **`train_test_split`**: Splits the processed dataset `X_final` and target `y` into training sets (80%) and testing sets (20%). Fixing `random_state=42` ensures our split is reproducible across different runs.
* **`LinearRegression()`**: Instantiates our learning algorithm, which assumes a linear relationship between our features and house prices.
* **`model.fit()`**: Trains the estimator on the training data (`X_train` and `y_train`) by finding the optimal weights (coefficients and intercept) that minimize prediction errors.

---

**4. Model Evaluation**

We generate predictions on the unseen test dataset (`X_test`) and evaluate model performance using RMSE and $R^2$ Score.

In [4]:
from sklearn.metrics import mean_squared_error, r2_score

# 1. Predict target values on unseen test set
y_pred = model.predict(X_test)

# 2. Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

# 3. Compare actual vs predicted values
comparison = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
print("\n--- Actual vs Predicted ---")
print(comparison)

--- Model Performance ---
Root Mean Squared Error (RMSE): $45,147.33
R² Score: 0.7742

--- Actual vs Predicted ---
   Actual      Predicted
0  190000  159929.952306
1  380000  436323.657972


### Explanation of Step 4: Model Evaluation
To measure how well our trained linear model generalizes to new, unseen data, we evaluate its predictions:
* **`model.predict()`**: Uses the trained model weights to predict house prices for the held-out test features (`X_test`).
* **Root Mean Squared Error (RMSE)**: Calculates the average magnitude of prediction error. It is computed in the same unit as our target ($ USD), penalizing larger errors more heavily.
* **R² (Coefficient of Determination) Score**: Measures how much of the variance in house prices is successfully explained by our model features. A score closer to 1.0 represents a better fit.
* **Actual vs. Predicted Comparison**: Creates a comparative table mapping the real actual house values from our test dataset alongside the model's predictions to visually inspect accuracy.